In [12]:
# Cargamos librerias
import cobra
import cometspy as c
import pandas as pd
from cobra import Reaction,Metabolite
import numpy as np
import matplotlib.pyplot as plt
import copy
import os
# Si no carga.....
os.environ['COMETS_HOME'] = r"C:\comets_windows\comets_2.12.5"
#os.environ['GUROBI_COMETS_HOME'] = "C:\\gurobi901\\win64" 

## Cargamos funciones

In [13]:
def ensure_sinks_are_not_exchanges(model):
    rxn_names = ['sink_PHAg','SK_pqqA_kt_c']
    for rxn in rxn_names:
        model.reactions.loc[model.reactions.REACTION_NAMES == rxn, 'EXCH'] = False
        model.reactions.loc[model.reactions.REACTION_NAMES == rxn, 'EXCH_IND'] = 0
def initialize_layout(models):
        #The layout is a file with the stablished format
    layout=c.layout()
    for model in models:
        ensure_sinks_are_not_exchanges(model)
        layout.add_model(model)
        layout.add_typical_trace_metabolites()
        layout.update_models()
    return(layout)
def make_df_and_graph(strains, metabolites, comets, max_cycles,filesufix='',show_plot=True):
    '''This function creates a figure and saves it to pdf format.
    It also creates the file biomass_vs_met.txt which contais the quantity
    of each strain and metabolite and has the following columns:
    time(h), strain1 ... strainX, met1 ... metX.'''
    file_name='_'.join(metabolites)
    df = comets.media #We get the media composition results'
    df_media=copy.deepcopy(df.loc[df['cycle']<max_cycles])
    df2=comets.total_biomass #We get the biomass results
    df_biomass=copy.deepcopy(df2.loc[df2['cycle']<max_cycles])
    columns=['cycle']
    for i in range(0,len(strains)):
        columns.append(strains[i])
    df_biomass.columns=columns
    
    """For each metabolite a column with all zeros is added to the dataframe and each row that contains a value
     (metabolite concentration)is changed in the dataframe"""
    for d in metabolites:
        columns.append(d)
        met =df_media.loc[df_media['metabolite'] == d]

        temp=np.zeros(max_cycles) #Create an array with all zeros
        df_biomass[d]=temp #We added it to the dataframe
        j=1
        while j < (max_cycles): #For each cycle
            if (met.cycle==j).any(): #If the row exists
                df_biomass.loc[j-1,d] = float(met[met['cycle']==j]['conc_mmol']) #Its dataframe value is changed
            j+=1
    df_biomass.columns=columns
    np.savetxt(r'biomass_vs_'+file_name+'_template.txt', df_biomass.values, fmt='%s',delimiter='\t',header='\t'.join(columns)) #The data is saved
    #---Starting the figure 
    plt.ioff()
    fig, ax = plt.subplots()

    ax.set_xlabel('time (h)')
    ax.set_ylabel('biomass (g/L)')
    c=['k', 'm', 'b', 'g', 'r']
    j=0
    for i in strains:
        ax.plot(df_biomass['cycle']*0.1, df_biomass[i], label=i, color=c[j])
        j+=1
    ax2 = ax.twinx()
    ax2.set_ylabel('metabolite conc (mM)')
    for m in metabolites:
        ax2.plot(df_biomass['cycle']*0.1, df_biomass[m], label=m)

    handles, labels = ax.get_legend_handles_labels()
    handle_list, label_list = [], []

    for handle, label in zip(handles, labels):
        if label not in label_list:
            handle_list.append(handle)
            label_list.append(label)
    handles, labels = ax2.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        if label not in label_list:
            handle_list.append(handle)
            label_list.append(label)
    plt.legend(handle_list, label_list)
    
    #saving the figure to a pdf
    plt.savefig('biomass_vs_'+file_name+filesufix+'_template_plot.pdf')
    if show_plot:
        plt.show()
    return df_biomass


def load_medium(layout):
    layout.set_specific_metabolite("glc__D_e",10)
    layout.set_specific_metabolite("o2_e",1000)
    layout.set_specific_metabolite("ca2_e",1000)
    layout.set_specific_metabolite("co2_e",100)
    layout.set_specific_metabolite("cobalt2_e",1000)
    layout.set_specific_metabolite("cu2_e",1000)
    layout.set_specific_metabolite("fe2_e",100)
    layout.set_specific_metabolite("fe3_e",100)
    layout.set_specific_metabolite("h_e",100)
    layout.set_specific_metabolite("h2o_e",1000)
    layout.set_specific_metabolite("k_e",100)
    layout.set_specific_metabolite("mg2_e",100)
    layout.set_specific_metabolite("mn2_e",100)
    layout.set_specific_metabolite("mobd_e",100)
    layout.set_specific_metabolite("na1_e",100)
    layout.set_specific_metabolite("no3_e",100)
    layout.set_specific_metabolite("pi_e",100)
    layout.set_specific_metabolite("so4_e",100)
    layout.set_specific_metabolite("zn2_e",1000)
    layout.set_specific_metabolite("nh4_e",40)
    layout.set_specific_metabolite("cl_e",1000)
    layout.set_specific_metabolite("ni2_e",1000)
    #layout.set_specific_metabolite("pqqA_kt_c",1000)

def intersection_exchange_metabolites(models):
    # Calculo del medio de varias especies
    all_exchanged_mets = []
    for m in models:
        all_exchanged_mets.extend(m.get_exchange_metabolites())
    all_exchanged_mets = sorted(list(set(list(all_exchanged_mets))))
    return(all_exchanged_mets)




def initialize_params(package, globall):
    """Function to initialize the comets' params class
    it can be initialize by submitting two files, one for the package parameters
    and one for the global ones.
    If you don't submit a file, the params class will be initialize with the values stated below
    which have been tested in this simulation"""
    
    if package and globall is not None:
        params = c.params(global_params = globall, package_params= package)
    elif package is not None:
        params = c.params(package_params=package)
    elif globall is not None:
        params = c.params(global_params=globall)
        
    else:
        params = c.params()
        params.all_params['maxCycles']=350 # horas * 10
        params.all_params['timeStep']=0.1
        params.all_params['spaceWidth']=0.05
        params.all_params['allowCellOverlap']= True
        params.all_params['cellSize'] = 4.3e-13
        params.all_params['deathRate']= 0.00
        params.all_params['dilFactor'] = 1e-2
        params.all_params['dilTime'] = 12
        params.all_params['numRunThreads']= 8
        params.all_params['maxSpaceBiomass']= 100
        params.all_params['defaultVmax']= 20
        params.all_params['defaultKm']= 0.01
        params.all_params['defaultHill']= 1
        params.all_params['showCycleTime']= True
        params.all_params['geneFractionalCost']= 0
        params.all_params['writeTotalBiomassLog']= True
        params.all_params['writeMediaLog']= True
        params.all_params['writeFluxLog']= True
        params.all_params['useLogNameTimeStamp']= False
        params.all_params['flowDiffRate']= 1e-7
        params.all_params['growthDiffRate']=1e-7
        params.all_params['FluxLogRate']= 1
        params.all_params['MediaLogRate']= 1
        params.all_params['numExRxnSubsteps']= 12
        params.all_params['geneFractionalCost'] = 0
        params.all_params['exchangestyle']= 'Standard FBA'#'Monod Style'
        params.all_params['biomassMotionStyle']= 'Diffusion 2D(Crank-Nicolson)'
        params.all_params['mutRate'] = 1e-9
        params.all_params['addRate'] = 1e-9
        params.all_params['minSpaceBiomass'] = 1e-10

        
    #check if param 'maxSpaceBiomass' has default value
    if (params.all_params['maxSpaceBiomass']== 0.1):
        print('The parameter "maxSpaceBiomass" is set to 0.1.\n' \
              'It may need to change if the mo used growths well.')

    return params

In [14]:
def simulate(model,model_inoculum, model_name,total_biomass, metabolite_list, show_plot=True):    

    model_comets1=c.model(model)
    model_comets1.id=model_name
    model_comets1.initial_pop=[0,0,model_inoculum]

    # Create the layout
    layout=c.layout([model_comets1])
    load_medium(layout)

    print("Loading the comets parameters...")
    params_package=None
    params_global=None
    params =initialize_params(params_package, params_global)
    params.all_params['exchangestyle']='Standard FBA'#'Monod Style'
    print("Creating the comets object...")
    comets = c.comets(layout, params)
    comets.run(delete_files=True)
    mydf=make_df_and_graph([model_name],  metabolite_list, comets, params.all_params['maxCycles']-1,filesufix=model_inoculum.__str__(),show_plot=show_plot)
    final_cycle=mydf.iloc[params.all_params['maxCycles']-2]
    return(mydf,final_cycle)

## Leemos los modelos

In [15]:
import cobra
# Ejemplo de uso
ruta = r"C:\Users\yanni\Desktop\CNB\Modelos\ΔlppA early_model_cutoff_0.135.json"
my_model=cobra.io.load_json_model(ruta)
my_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,4.737E-06,0,0.00%
cl_e,EX_cl_e,4.737E-06,0,0.00%
cobalt2_e,EX_cobalt2_e,3.158E-06,0,0.00%
cu2_e,EX_cu2_e,3.158E-06,0,0.00%
fe2_e,EX_fe2_e,7.329E-06,0,0.00%
glc__D_e,EX_glc__D_e,0.5195,6,100.00%
k_e,EX_k_e,0.0001776,0,0.00%
mg2_e,EX_mg2_e,7.895E-06,0,0.00%
mn2_e,EX_mn2_e,3.158E-06,0,0.00%
mobd_e,EX_mobd_e,3.158E-06,0,0.00%


In [ ]:
import os

# Cambiamos el ID interno para que no tenga espacios ni el símbolo Delta
my_model.id = "ΔlppA_early_model"

simulate(my_model, # modelo a estudiar que he puesto en la anterior celda
         2.2e7, #inoculo inicial
         'ΔlppA_early', # nombre del modelo
         total_biomass=0.1, # no tocar
         metabolite_list=['ac_e', 'for_e', 'lac__D_e', 'glc__D_e',], # metabolitos de interes
         show_plot=True)

Loading the comets parameters...
Creating the comets object...

Running COMETS simulation ...
Error: COMETS simulation did not complete

     examine comets.run_output for the full java trace

     if we detect a common reason, it will be stated in the RuntimeError at the bottom


RuntimeError: COMETS simulation did not complete:
 undetected reason. examine comets.run_output for JAVA trace

In [17]:
# Esto nos dirá qué le pasó a Java internamente
print("--- COMIENZO DEL RASTRO DE JAVA ---")
try:
    # Intentamos acceder al objeto 'comets' que se creó dentro de tu función
    # Si la función falló en la línea 18, el objeto debería existir en el espacio global
    import inspect
    # Buscamos el objeto comets en el frame anterior si simulate falló
    print("".join(comets.run_output))
except Exception as e:
    print("No se pudo extraer el log automáticamente.")
    print("Prueba a ejecutar: print(''.join(sim.run_output)) si llamaste a la variable sim.")

--- COMIENZO DEL RASTRO DE JAVA ---
No se pudo extraer el log automáticamente.
Prueba a ejecutar: print(''.join(sim.run_output)) si llamaste a la variable sim.


In [18]:
import os
import cometspy as c

# 1. Configuración de rutas manual
os.environ['COMETS_HOME'] = r"C:\comets_windows\comets_2.12.5"
gurobi_bin = r"C:\gurobi1301\win64\bin"
if gurobi_bin not in os.environ['PATH']:
    os.environ['PATH'] = gurobi_bin + os.pathsep + os.environ['PATH']

# 2. Creamos un objeto de simulación mínimo para forzar el error
try:
    # Usamos el modelo que ya tienes cargado
    my_model.id = "test_lppA" 
    test_model = c.model(my_model)
    test_layout = c.layout(test_model)
    test_params = c.params()
    test_params.set_param('maxCycles', 2) # Solo 2 ciclos para que sea instantáneo

    # Intentamos correrlo fuera de tu función 'simulate'
    sim_debug = c.comets(test_layout, test_params)
    sim_debug.run()
    print("✅ ¡Increíble! Fuera de la función sí ha funcionado.")

except Exception:
    print("\n--- ¡CAPTURADO! ESTE ES EL ERROR DE JAVA ---")
    # Esto es lo que necesito que me copies y pegues
    print("".join(sim_debug.run_output)) 
    print("------------------------------------------")


Running COMETS simulation ...
Error: COMETS simulation did not complete

     examine comets.run_output for the full java trace

     if we detect a common reason, it will be stated in the RuntimeError at the bottom

--- ¡CAPTURADO! ESTE ES EL ERROR DE JAVA ---
'\\wsl.localhost\Ubuntu\home\yannis\CNB\CNB\Scripts\Python'
CMD.EXE se inici con esta ruta como el directorio actual. No se permiten
rutas UNC. Regresando de manera predeterminada al directorio Windows.
-script
running script file: \\wsl.localhost\Ubuntu\home\yannis\CNB\CNB\Scripts\Python/.current_script_0x1e264c7e500
Current Java version: 26
Error occurred while loading parameter file:
.current_global_0x1e264c7e500 (El sistema no puede encontrar el archivo especificado)
MAX CYCLES = -1
Error occurred while loading parameter file:
.current_package_0x1e264c7e500 (El sistema no puede encontrar el archivo especificado)
MAX CYCLES = -1
Loading layout file '.current_layout_0x1e264c7e500'...
null\COMETS_manifest.txt (El sistema no pu